In [124]:
!pip install -q pytorch-metric-learning

In [125]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

import torchvision
from pytorch_metric_learning.losses import NTXentLoss

from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd

In [126]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

EPOCHS = 200
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
TEMP = 0.4

In [127]:
train_transformation = torchvision.transforms.Compose([
                                        torchvision.transforms.ToTensor()])

test_transformation = torchvision.transforms.Compose([
                                        torchvision.transforms.ToTensor(), 
                                        torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])


simclr_aug = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(32, scale=(0.2, 1.0), ratio=(3/4, 4/3)),
    torchvision.transforms.RandomHorizontalFlip(p=0.5),
    torchvision.transforms.RandomApply([
        torchvision.transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2)], p=0.8),
    torchvision.transforms.RandomGrayscale(p=0.2),
    torchvision.transforms.RandomApply([
        torchvision.transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5),
    #torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])


class TwoViews():
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)


def initialize_weights(model):
    for m in model.modules():

        if isinstance(m, torch.nn.Conv2d):
            torch.nn.init.kaiming_normal_(
                m.weight,
                mode='fan_out',
                nonlinearity='relu'
            )
            if m.bias is not None:
                torch.nn.init.zeros_(m.bias)

        elif isinstance(m, torch.nn.BatchNorm2d):
            torch.nn.init.ones_(m.weight)
            torch.nn.init.zeros_(m.bias)

        elif isinstance(m, torch.nn.Linear):
            torch.nn.init.kaiming_normal_(
                m.weight,
                mode='fan_out',
                nonlinearity='relu'
            )
            if m.bias is not None:
                torch.nn.init.zeros_(m.bias)

In [128]:
class CNNModel(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv_1 = torch.nn.Conv2d(in_channels=in_channels, out_channels=16, kernel_size=2, stride=2)
        self.bn_1 = torch.nn.BatchNorm2d(16)
        self.conv_2 = torch.nn.Conv2d(in_channels=16, out_channels=32, kernel_size=2, stride=2)
        self.bn_2 = torch.nn.BatchNorm2d(32)
        self.conv_3 = torch.nn.Conv2d(in_channels=32, out_channels=32, kernel_size=2)
        self.bn_3 = torch.nn.BatchNorm2d(32)

        self.act_func = torch.nn.ReLU()

        # projection head
        self.projection_enc = torch.nn.Linear(32 * 7 * 7, 32)

    def encode(self, x):
        """Backbone features (pre-projection). Use this for downstream tasks / eval."""
        x = self.act_func(self.bn_1(self.conv_1(x)))
        x = self.act_func(self.bn_2(self.conv_2(x)))
        x = self.act_func(self.bn_3(self.conv_3(x)))
        return x

    def forward(self, x):
        """Encode a single (already-augmented) view and project it."""
        features = self.encode(x)
        return self.projection_enc(features.reshape(-1, 32 * 7 * 7))

In [129]:
model = CNNModel(in_channels=3)
model.apply(initialize_weights)
model = model.to(device)

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transformation)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)

augment_cls = TwoViews(simclr_aug)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = NTXentLoss(temperature=TEMP)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
model.train()
loss_history = []

for epoch in range(EPOCHS):
    epoch_losses = []

    for data, _ in trainloader:
        optimizer.zero_grad()

        view_1, view_2 = augment_cls(data)

        view_1 = view_1.to(device)
        view_2 = view_2.to(device)

        # project each view independently
        z1 = model(view_1)
        z2 = model(view_2)

        embeddings = torch.cat((z1, z2), dim=0)
        # positives share the same label: view_1[i] and view_2[i]
        indices = torch.arange(z1.size(0), device=z1.device)
        labels = torch.cat((indices, indices), dim=0)

        loss = criterion(embeddings, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        epoch_losses.append(loss.item())

    scheduler.step()
    mean_loss = sum(epoch_losses) / len(epoch_losses)
    loss_history.append(mean_loss)
    print(f"Epoch: {epoch+1}, mean loss: {mean_loss:.4f}")

Epoch: 1, mean loss: 4.5371
Epoch: 2, mean loss: 4.3942
Epoch: 3, mean loss: 4.2703
Epoch: 4, mean loss: 4.2608


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Mean NT-Xent loss")
plt.title("SimCLR contrastive pretraining")
plt.grid(True)
plt.show()

In [ ]:
def label_points(x, y, val, ax):
    a = pd.concat({"x":x, "y":y, "val":val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point["x"]+0.02, point["y"], str(int(point["val"])))

In [ ]:
sample, label = next(iter(trainloader))

emb_h = model.encode(sample.to(device))
emb_h = emb_h.cpu().detach().numpy()
emb_h = emb_h.reshape(-1, 32*7*7)

emb_h_2D = TSNE(n_components=2, learning_rate="auto", init="random").fit_transform(emb_h)

ax = sns.scatterplot(x=emb_h_2D[:, 0], y=emb_h_2D[:, 1], hue=label, alpha=0.5, palette="tab10")
# add numbers to the samples
annotations = list(range(len(emb_h_2D[:, 0])))
label_points(pd.Series(emb_h_2D[:, 0]), pd.Series(emb_h_2D[:, 1]), pd.Series(annotations), ax)

In [ ]:
def similar_emb(a, b, eps=1e-8):
    a_n, b_n = np.linalg.norm(a, axis=1)[:, None], np.linalg.norm(b, axis=1)[:, None]
    a_norm = a/np.maximum(a_n, eps*np.ones_like(a_n))
    b_norm = b/np.maximum(b_n, eps*np.ones_like(b_n))

    sim_emb = np.matmul(a_norm, b_norm.T)

    return sim_emb
    

In [ ]:
similarity = similar_emb(emb_h, emb_h)
max_indices = np.argsort(similarity, axis=1)[:, -3:][:, ::-1]

In [ ]:
print(max_indices)

In [ ]:
sample = sample.permute(0, 2, 3, 1)
print(sample.shape)

In [ ]:
plt.imshow(sample[1])

In [ ]:
plt.imshow(sample[14])